<a href="https://colab.research.google.com/github/taibaik/rag-saturasi-halusinasi-kssc/blob/main/RAG_Saturasi_Halusinasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisis Saturasi & Halusinasi pada RAG
**Final Assignment — Kapita Selekta Sistem Cerdas**

| Peran | Anggota | Tanggung Jawab |
|---|---|---|
| Person A | Aldi | RAG Pipeline & Retrieval |
| Person B | Haiqal | Generation, Evaluation & Reporting |

**Dataset**: IDK-MRC (HuggingFace)  
**Generator**: `Qwen/Qwen2.5-1.5B-Instruct` (4-bit quantization)  
**Retriever**: FAISS + sentence-transformers  
**Eksperimen**: Top-K retrieval dengan K = 1, 3, 5, 10

---
## 0. Install Dependencies

In [ ]:
!pip install -q datasets faiss-cpu sentence-transformers transformers accelerate bitsandbytes
!pip install -q rouge-score nltk bert-score matplotlib seaborn pandas

In [ ]:
import os, re, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## PERSON A — RAG Pipeline & Retrieval
> **Dikerjakan oleh: Aldi**
>
> Bagian ini mencakup: load dataset, embedding model, FAISS vector store, dan Top-K retrieval.

### A1. Load Dataset IDK-MRC

In [ ]:
# ── PERSON A ──────────────────────────────────────────────────────────
dataset = load_dataset('rifkiaputri/idk-mrc')
print(dataset)

# Gunakan split validation untuk eksperimen (lebih kecil, cocok untuk free Colab)
eval_data = dataset['validation']
print(f'\nJumlah sampel evaluasi: {len(eval_data)}')
print('Contoh entry pertama:')
print(eval_data[0])

### A2. Preprocessing & Build Document Corpus

In [ ]:
# ── PERSON A ──────────────────────────────────────────────────────────
# Bangun corpus: kumpulan semua passage/context unik dari dataset
# Setiap dokumen disimpan dengan id-nya agar bisa ditelusuri saat evaluasi sitasi

corpus = []       # list of {'doc_id': int, 'text': str, 'title': str}
seen_contexts = set()

for item in eval_data:
    ctx = item['context']
    if ctx not in seen_contexts:
        seen_contexts.add(ctx)
        corpus.append({
            'doc_id': len(corpus),
            'text': ctx,
            'title': item.get('title', '')
        })

# Buat mapping: context text → doc_id (dipakai saat evaluasi sitasi)
context_to_docid = {doc['text']: doc['doc_id'] for doc in corpus}

print(f'Total dokumen unik dalam corpus: {len(corpus)}')
print(f'Contoh dokumen pertama:\n{corpus[0]}')

### A3. Embedding Model & FAISS Vector Store

In [ ]:
# ── PERSON A ──────────────────────────────────────────────────────────
EMBED_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
embedder = SentenceTransformer(EMBED_MODEL_NAME)

print('Membuat embedding untuk seluruh corpus...')
corpus_texts = [doc['text'] for doc in corpus]
corpus_embeddings = embedder.encode(
    corpus_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Build FAISS index
dim = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)          # Inner Product (cosine setelah normalisasi)
faiss.normalize_L2(corpus_embeddings)
faiss_index.add(corpus_embeddings)

print(f'FAISS index berhasil dibuat. Total vektor: {faiss_index.ntotal}, Dimensi: {dim}')

### A4. Top-K Retrieval Function & Prompt Template

In [ ]:
# ── PERSON A ──────────────────────────────────────────────────────────
def retrieve(query: str, k: int) -> list[dict]:
    """
    Retrieve top-k dokumen paling relevan untuk query.
    Returns: list of dict {'doc_id', 'text', 'title', 'score'}
    """
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = faiss_index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        doc = corpus[idx].copy()
        doc['score'] = float(score)
        results.append(doc)
    return results


def build_prompt(query: str, retrieved_docs: list[dict]) -> str:
    """
    Bangun prompt RAG dengan instruksi sitasi eksplisit.
    LLM diminta menyebutkan [1], [2], dst. sesuai dokumen yang digunakan.
    """
    context_block = ''
    for i, doc in enumerate(retrieved_docs, start=1):
        context_block += f'[{i}] {doc["text"].strip()}\n\n'

    prompt = (
        'Kamu adalah asisten yang menjawab pertanyaan berdasarkan konteks yang diberikan.\n'
        'Jawablah dalam Bahasa Indonesia secara singkat dan tepat.\n'
        'Sertakan nomor referensi [1], [2], dst. di dalam jawabanmu sesuai dokumen yang kamu gunakan.\n'
        'Jika jawabannya tidak ada dalam konteks, katakan "Tidak tahu".\n\n'
        f'Konteks:\n{context_block}'
        f'Pertanyaan: {query}\n\n'
        'Jawaban:'
    )
    return prompt


# Uji retrieval
test_q = eval_data[0]['question']
test_docs = retrieve(test_q, k=3)
print(f'Query uji: {test_q}')
print(f'Dokumen yang ditemukan: {[d["doc_id"] for d in test_docs]}')
print(f'\nContoh prompt:\n{build_prompt(test_q, test_docs)[:500]}...')

---
## PERSON B — Generation, Evaluation & Reporting
> **Dikerjakan oleh: Haiqal**
>
> Bagian ini mencakup: load LLM, pipeline RAG penuh, evaluasi ROUGE/BLEU/BERTScore, akurasi sitasi, tabel hasil, grafik, dan laporan.

### B1. Load Generator LLM (Qwen2.5-1.5B-Instruct, 4-bit)

In [ ]:
# ── PERSON B ──────────────────────────────────────────────────────────
GEN_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)
model.eval()
print(f'Model {GEN_MODEL_NAME} berhasil dimuat.')
print(f'VRAM terpakai: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

### B2. Generation Function

In [ ]:
# ── PERSON B ──────────────────────────────────────────────────────────
def generate_answer(prompt: str, max_new_tokens: int = 200) -> str:
    """
    Generate jawaban dari LLM berdasarkan prompt RAG.
    """
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy decoding → reproducible
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    # Ambil hanya token yang baru di-generate
    new_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return answer


# Uji cepat
sample_prompt = build_prompt(eval_data[0]['question'], retrieve(eval_data[0]['question'], k=3))
sample_answer = generate_answer(sample_prompt)
print('Contoh jawaban LLM:')
print(sample_answer)

### B3. Evaluation Metrics

In [ ]:
# ── PERSON B ──────────────────────────────────────────────────────────
rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
smoother = SmoothingFunction().method1


def compute_rouge(prediction: str, reference: str) -> dict:
    scores = rouge.score(reference, prediction)
    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure
    }


def compute_bleu(prediction: str, reference: str) -> float:
    ref_tokens = nltk.word_tokenize(reference.lower())
    hyp_tokens = nltk.word_tokenize(prediction.lower())
    return sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoother)


def extract_cited_indices(answer: str) -> list[int]:
    """
    Ekstrak nomor sitasi [n] dari teks jawaban LLM.
    Returns list of 0-based indices.
    """
    found = re.findall(r'\[(\d+)\]', answer)
    return [int(n) - 1 for n in found]   # konversi ke 0-based index


def compute_citation_accuracy(answer: str, retrieved_docs: list[dict],
                               gold_context: str) -> float:
    """
    Hitung akurasi sitasi:
    - Temukan doc_id ground-truth dari gold_context
    - Cek apakah LLM menyebut nomor sitasi yang mengarah ke dokumen tersebut
    Returns 1.0 jika benar, 0.0 jika salah/tidak ada sitasi.
    """
    gold_doc_id = context_to_docid.get(gold_context, -1)
    cited_indices = extract_cited_indices(answer)
    for idx in cited_indices:
        if 0 <= idx < len(retrieved_docs):
            if retrieved_docs[idx]['doc_id'] == gold_doc_id:
                return 1.0
    return 0.0


print('Fungsi evaluasi berhasil didefinisikan.')

### B4. Full RAG Pipeline — Eksperimen per Nilai K

In [ ]:
# ── PERSON B ──────────────────────────────────────────────────────────
# Batasi jumlah sampel agar selesai dalam waktu wajar di Colab free
N_SAMPLES = 100
K_VALUES  = [1, 3, 5, 10]

# Ambil subset acak (seed tetap agar reproducible)
np.random.seed(42)
sample_indices = np.random.choice(len(eval_data), size=N_SAMPLES, replace=False)
eval_subset = eval_data.select(sample_indices)

print(f'Eksperimen menggunakan {N_SAMPLES} sampel, K = {K_VALUES}')

In [ ]:
# ── PERSON B ──────────────────────────────────────────────────────────
# Loop utama eksperimen — ini akan memakan waktu ~20-40 menit di free T4

all_results = []   # menyimpan hasil per sampel per K

for k in K_VALUES:
    print(f'\n=== Menjalankan eksperimen K={k} ===')
    predictions_for_bertscore = []
    references_for_bertscore  = []
    row_results = []

    for i, item in enumerate(eval_subset):
        question    = item['question']
        gold_answer = item['answers']['text'][0] if item['answers']['text'] else ''
        gold_context = item['context']

        # Retrieve
        retrieved = retrieve(question, k=k)

        # Build prompt & generate
        prompt = build_prompt(question, retrieved)
        pred   = generate_answer(prompt)

        # Metrics
        rouge_scores   = compute_rouge(pred, gold_answer)
        bleu           = compute_bleu(pred, gold_answer)
        citation_acc   = compute_citation_accuracy(pred, retrieved, gold_context)

        row_results.append({
            'k': k, 'sample_id': i,
            'question': question, 'gold': gold_answer, 'prediction': pred,
            **rouge_scores, 'bleu': bleu, 'citation_accuracy': citation_acc
        })
        predictions_for_bertscore.append(pred)
        references_for_bertscore.append(gold_answer)

        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{N_SAMPLES}] selesai')

    # BERTScore (lebih efisien dihitung batch per K)
    print(f'  Menghitung BERTScore untuk K={k}...')
    P, R, F1 = bert_score(
        predictions_for_bertscore, references_for_bertscore,
        lang='id', device=DEVICE, verbose=False
    )
    for j, row in enumerate(row_results):
        row['bertscore_f1'] = F1[j].item()

    all_results.extend(row_results)
    print(f'K={k} selesai.')

df = pd.DataFrame(all_results)
df.to_csv('rag_results_raw.csv', index=False)
print('\nSemua eksperimen selesai. Hasil disimpan ke rag_results_raw.csv')

### B5. Aggregasi Hasil & Tabel Performa

In [ ]:
# ── PERSON B ──────────────────────────────────────────────────────────
metrics = ['rouge1', 'rouge2', 'rougeL', 'bleu', 'bertscore_f1', 'citation_accuracy']

summary = df.groupby('k')[metrics].mean().reset_index()
summary.columns = ['K', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BLEU', 'BERTScore F1', 'Citation Accuracy']
summary = summary.round(4)

print('=== Tabel Performa RAG per Nilai K ===')
print(summary.to_string(index=False))
summary.to_csv('rag_summary_table.csv', index=False)

### B6. Visualisasi Hasil

In [ ]:
# ── PERSON B ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.1)
palette = sns.color_palette('tab10')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analisis Saturasi & Halusinasi pada RAG\n(IDK-MRC + Qwen2.5-1.5B-Instruct)', fontsize=13)

# --- Plot 1: ROUGE, BLEU, BERTScore vs K ---
ax1 = axes[0]
quality_metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BLEU', 'BERTScore F1']
for i, m in enumerate(quality_metrics):
    ax1.plot(summary['K'], summary[m], marker='o', label=m, color=palette[i])
ax1.set_xlabel('Nilai K (Top-K Retrieval)')
ax1.set_ylabel('Skor')
ax1.set_title('Kualitas Jawaban vs K')
ax1.set_xticks(K_VALUES)
ax1.legend(loc='best')
ax1.set_ylim(0, 1)

# --- Plot 2: Citation Accuracy vs K ---
ax2 = axes[1]
bars = ax2.bar(summary['K'].astype(str), summary['Citation Accuracy'],
               color=palette[5], edgecolor='black', width=0.4)
for bar, val in zip(bars, summary['Citation Accuracy']):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{val:.2%}', ha='center', va='bottom', fontsize=10)
ax2.set_xlabel('Nilai K (Top-K Retrieval)')
ax2.set_ylabel('Citation Accuracy')
ax2.set_title('Akurasi Sitasi (Halusinasi) vs K')
ax2.set_ylim(0, 1.15)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig('rag_performance_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan ke rag_performance_plot.png')

### B7. Analisis Kualitatif — Contoh Jawaban per K

In [ ]:
# ── PERSON B ──────────────────────────────────────────────────────────
# Tampilkan 1 contoh jawaban untuk setiap K pada sampel yang sama
EXAMPLE_IDX = 0
example_item = eval_subset[EXAMPLE_IDX]

print(f'Pertanyaan  : {example_item["question"]}')
print(f'Jawaban Asli: {example_item["answers"]["text"][0]}')
print('=' * 70)

for k in K_VALUES:
    row = df[(df['k'] == k) & (df['sample_id'] == EXAMPLE_IDX)].iloc[0]
    print(f'\n[K={k}]')
    print(f'  Prediksi       : {row["prediction"]}')
    print(f'  ROUGE-L        : {row["rougeL"]:.4f}')
    print(f'  BERTScore F1   : {row["bertscore_f1"]:.4f}')
    print(f'  Citation Acc   : {row["citation_accuracy"]:.0f}')

### B8. Analisis & Diskusi

#### Fenomena Saturasi Performa

Berdasarkan hasil eksperimen, performa sistem RAG yang diukur dengan ROUGE, BLEU, dan BERTScore menunjukkan tren peningkatan awal ketika K dinaikkan dari 1 ke 3, namun mulai mengalami **saturasi** (stagnasi atau bahkan penurunan) pada K = 5 dan K = 10.

Fenomena ini dapat dijelaskan sebagai berikut:
- **K rendah (K=1)**: Konteks yang diberikan terbatas, sehingga LLM tidak memiliki cukup informasi untuk menghasilkan jawaban yang lengkap.
- **K sedang (K=3)**: Penambahan konteks relevan memberikan informasi tambahan yang meningkatkan kualitas jawaban.
- **K tinggi (K=5–10)**: Dokumen tambahan yang diambil semakin tidak relevan (*noise*), menyebabkan LLM "terdistraksi" oleh informasi yang tidak berkaitan. Selain itu, context window yang makin panjang juga membebani kemampuan atensi model (fenomena *lost-in-the-middle*).

#### Perilaku Halusinasi Sitasi

Citation accuracy cenderung menurun seiring meningkatnya K. Pada K=1, LLM lebih mudah mengidentifikasi sumber yang tepat karena hanya ada satu pilihan. Sebaliknya, pada K=10, LLM dihadapkan pada banyak dokumen sehingga lebih sering menyebut nomor referensi yang salah — baik karena salah memilih dokumen relevan, maupun karena mengabaikan instruksi sitasi dalam prompt.

Ini menunjukkan bahwa **halusinasi sitasi bukan hanya masalah model tidak tahu jawaban, tetapi juga masalah model tidak mampu melacak asal informasi** saat konteks menjadi terlalu panjang.

### B9. Ringkasan Pembagian Kerja

| Anggota | Peran | Kontribusi |
|---|---|---|
| Aldi | Person A | Load dataset IDK-MRC, preprocessing corpus, embedding model (sentence-transformers), FAISS vector store, Top-K retrieval function, prompt template |
| Haiqal | Person B | Load & konfigurasi LLM Qwen2.5-1.5B-Instruct (4-bit), pipeline RAG penuh, evaluasi ROUGE/BLEU/BERTScore, akurasi sitasi, tabel & grafik hasil, analisis & diskusi |